# 🔥 XAI + Counterfactual Explanation System
## Loan Approval Prediction

---
**Methods Used:**
- 📊 **SHAP** — Global + Local Feature Importance
- 🍋 **LIME** — Local Interpretable Model Explanations  
- 🎲 **DiCE** — Counterfactual Actionable Explanations

**Dataset:** [Kaggle Loan Prediction Dataset](https://www.kaggle.com/datasets/altruistdelhite04/loan-prediction-problem-dataset)

---

## 📦 Step 1: Install Required Libraries

In [ ]:
!pip install shap lime dice-ml scikit-learn pandas numpy matplotlib seaborn -q

## 📚 Step 2: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

import shap
import lime
import lime.lime_tabular
import dice_ml
from dice_ml import Dice

print('✅ All libraries imported successfully!')

## 📂 Step 3: Load Dataset
> ⚠️ **Note:** Kaggle se `train.csv` download karke is notebook ke same folder mein rakho
> 
> Link: https://www.kaggle.com/datasets/altruistdelhite04/loan-prediction-problem-dataset

In [ ]:
df = pd.read_csv('train.csv')

print('Dataset Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
df.head(10)

In [ ]:
# Missing values check
print('🔍 Missing Values:')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
display(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Dataset info
df.info()

## 🧹 Step 4: Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('📊 Loan Dataset - EDA', fontsize=16, fontweight='bold')

# Loan Status Distribution
df['Loan_Status'].value_counts().plot(kind='bar', ax=axes[0,0], color=['#e74c3c','#2ecc71'])
axes[0,0].set_title('Loan Status Distribution')
axes[0,0].set_xlabel('')
axes[0,0].tick_params(rotation=0)

# Gender vs Loan Status
pd.crosstab(df['Gender'], df['Loan_Status']).plot(kind='bar', ax=axes[0,1], color=['#e74c3c','#2ecc71'])
axes[0,1].set_title('Gender vs Loan Status')
axes[0,1].tick_params(rotation=0)

# Education vs Loan Status
pd.crosstab(df['Education'], df['Loan_Status']).plot(kind='bar', ax=axes[0,2], color=['#e74c3c','#2ecc71'])
axes[0,2].set_title('Education vs Loan Status')
axes[0,2].tick_params(rotation=0)

# Applicant Income Distribution
df['ApplicantIncome'].hist(ax=axes[1,0], bins=30, color='#3498db', edgecolor='black')
axes[1,0].set_title('Applicant Income Distribution')

# Loan Amount Distribution
df['LoanAmount'].hist(ax=axes[1,1], bins=30, color='#9b59b6', edgecolor='black')
axes[1,1].set_title('Loan Amount Distribution')

# Credit History vs Loan Status
pd.crosstab(df['Credit_History'], df['Loan_Status']).plot(kind='bar', ax=axes[1,2], color=['#e74c3c','#2ecc71'])
axes[1,2].set_title('Credit History vs Loan Status')
axes[1,2].tick_params(rotation=0)

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA Complete!')

## 🔧 Step 5: Data Preprocessing

In [ ]:
# Drop Loan_ID
df.drop('Loan_ID', axis=1, inplace=True)

# Fill missing values
df['Gender'].fillna(df['Gender'].mode()[0], inplace=True)
df['Married'].fillna(df['Married'].mode()[0], inplace=True)
df['Dependents'].fillna(df['Dependents'].mode()[0], inplace=True)
df['Self_Employed'].fillna(df['Self_Employed'].mode()[0], inplace=True)
df['LoanAmount'].fillna(df['LoanAmount'].median(), inplace=True)
df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].mode()[0], inplace=True)
df['Credit_History'].fillna(df['Credit_History'].mode()[0], inplace=True)

print('✅ Missing values filled!')
print('Remaining missing:', df.isnull().sum().sum())

In [ ]:
# Encode categorical columns
le = LabelEncoder()
cat_cols = ['Gender', 'Married', 'Dependents', 'Education',
            'Self_Employed', 'Property_Area', 'Loan_Status']

for col in cat_cols:
    df[col] = le.fit_transform(df[col])

print('✅ Encoding complete!')
df.head()

## 🤖 Step 6: Train ML Model (Random Forest)

In [ ]:
X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f'✅ Model Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%')
print('\n📋 Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Denied', 'Approved']))

In [ ]:
# Feature Importance Plot
feat_imp = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
feat_imp.plot(kind='barh', color='#3498db')
plt.title('🌲 Random Forest - Feature Importance', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

## 📊 Step 7: SHAP Explanation
> SHAP batata hai ki **har feature ne prediction pe kitna aur kaise effect dala**

In [ ]:
# SHAP Explainer
explainer_shap = shap.TreeExplainer(model)
shap_values = explainer_shap.shap_values(X_test)

print('✅ SHAP values computed!')
print(f'Shape: {np.array(shap_values).shape}')

In [ ]:
# Global Feature Importance - Summary Plot
plt.figure()
shap.summary_plot(shap_values[1], X_test, feature_names=feature_names, show=False)
plt.title('📊 SHAP - Global Feature Importance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ SHAP Summary Plot saved!')

In [ ]:
# Single Instance - Denied Case
denied_idx = X_test[y_pred == 0].index[0]
denied_pos = X_test.index.get_loc(denied_idx)

print(f'🔍 Explaining Denied Case (Index: {denied_idx})')
print(f'Prediction: ❌ DENIED')
print(f'Confidence: {model.predict_proba(X_test.iloc[[denied_pos]])[0][0]*100:.1f}%')
print()
display(X_test.iloc[[denied_pos]])

In [ ]:
# SHAP Force Plot for denied case
shap.force_plot(
    explainer_shap.expected_value[1],
    shap_values[1][denied_pos],
    X_test.iloc[denied_pos],
    feature_names=feature_names,
    matplotlib=True
)
plt.title('SHAP Force Plot - Denied Case', fontsize=12)
plt.tight_layout()
plt.savefig('shap_force.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Bar Plot - Top features for this instance
sv = shap_values[1][denied_pos]
shap_df = pd.DataFrame({
    'Feature': feature_names,
    'SHAP Value': sv
}).sort_values('SHAP Value', key=abs, ascending=True)

colors = ['#e74c3c' if x < 0 else '#2ecc71' for x in shap_df['SHAP Value']]

plt.figure(figsize=(10, 6))
plt.barh(shap_df['Feature'], shap_df['SHAP Value'], color=colors)
plt.axvline(x=0, color='black', linewidth=0.8)
plt.title('📊 SHAP Values - Denied Instance', fontsize=14, fontweight='bold')
plt.xlabel('SHAP Value (impact on prediction)')
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150)
plt.show()

print('\n🔴 Negative SHAP = Feature pushed toward DENIAL')
print('🟢 Positive SHAP = Feature pushed toward APPROVAL')

## 🍋 Step 8: LIME Explanation
> LIME locally ek simple model banata hai aur explain karta hai ki **is specific prediction ke liye kya important tha**

In [ ]:
# LIME Explainer
explainer_lime = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=feature_names,
    class_names=['Denied', 'Approved'],
    mode='classification'
)

lime_exp = explainer_lime.explain_instance(
    data_row=X_test.iloc[denied_pos].values,
    predict_fn=model.predict_proba,
    num_features=8
)

print('✅ LIME Explanation computed!')
print('\n🍋 LIME - Top Reasons for DENIAL:')
for feature, weight in lime_exp.as_list():
    direction = '🟢 Positive' if weight > 0 else '🔴 Negative'
    print(f'  {direction} | {feature}: {weight:.4f}')

In [ ]:
# LIME Visualization
lime_exp.as_pyplot_figure()
plt.title('🍋 LIME - Local Feature Importance (Denied Case)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('lime_explanation.png', dpi=150, bbox_inches='tight')
plt.show()

## 🎲 Step 9: DiCE Counterfactual Explanation
> DiCE batata hai: **"Kya change karo toh loan approve ho jata?"**
> 
> Example: *"Agar credit score 0 → 1 hota, loan approve ho jata"*

In [ ]:
# Prepare data for DiCE
df_dice = X_train.copy()
df_dice['Loan_Status'] = y_train.values

continuous_features = ['ApplicantIncome', 'CoapplicantIncome',
                       'LoanAmount', 'Loan_Amount_Term']

dice_data = dice_ml.Data(
    dataframe=df_dice,
    continuous_features=continuous_features,
    outcome_name='Loan_Status'
)

dice_model = dice_ml.Model(model=model, backend='sklearn')
dice_exp = Dice(dice_data, dice_model, method='random')

print('✅ DiCE Explainer ready!')

In [ ]:
# Generate Counterfactuals
query_instance = X_test.iloc[[denied_pos]]

print('📋 Original Instance (DENIED ❌):')
display(query_instance)

counterfactuals = dice_exp.generate_counterfactuals(
    query_instance,
    total_CFs=3,
    desired_class='opposite'
)

print('\n🔄 Counterfactual Suggestions (Changes needed for APPROVAL ✅):')

In [ ]:
# Show only changes
counterfactuals.visualize_as_dataframe(show_only_changes=True)

In [ ]:
# Pretty print counterfactual changes
cf_df = counterfactuals.cf_examples_list[0].final_cfs_df
original = query_instance.copy()

print('\n📌 Actionable Changes Required:')
print('='*55)
for col in feature_names:
    orig_val = original[col].values[0]
    cf_val = cf_df[col].values[0]
    if round(orig_val, 2) != round(cf_val, 2):
        print(f'  🔄 {col}')
        print(f'     Before: {orig_val:.2f}  →  After: {cf_val:.2f}')
        print()
print('='*55)
print('  ✅ Result: LOAN APPROVED!')

## 📋 Step 10: Comparison - SHAP vs LIME vs Counterfactual

In [ ]:
comparison_data = {
    'Method': ['SHAP', 'LIME', 'DiCE Counterfactual'],
    'Type': ['Global + Local', 'Local Only', 'Actionable'],
    'Explains': [
        'Har feature ka overall aur individual impact',
        'Is specific prediction ke liye feature importance',
        'Kya change karne se decision badlega'
    ],
    'Best For': [
        'Model debugging, feature selection',
        'Single prediction explanation',
        'User guidance, actionable insights'
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print('📊 Comparison Table:')
display(comparison_df)

## 🚀 Step 11: All-in-One `explain_decision()` Function

In [ ]:
def explain_decision(instance_idx):
    """
    Ek function jo SHAP + LIME + Counterfactual
    ek saath explain kare for any test instance
    
    Args:
        instance_idx: X_test mein position (0, 1, 2, ...)
    """
    instance = X_test.iloc[[instance_idx]]
    prediction = model.predict(instance)[0]
    probability = model.predict_proba(instance)[0]

    print('='*60)
    print('🏦 LOAN DECISION EXPLANATION SYSTEM')
    print('='*60)
    print(f'Decision  : {"✅ APPROVED" if prediction == 1 else "❌ DENIED"}')
    print(f'Confidence: {max(probability)*100:.1f}%')
    print()
    display(instance)

    # ── SHAP ──────────────────────────────────────
    print('\n' + '-'*40)
    print('📊 SHAP - Top 3 Reasons:')
    print('-'*40)
    sv = explainer_shap.shap_values(instance)[1][0]
    shap_df = pd.DataFrame({
        'Feature': feature_names,
        'SHAP Value': sv
    }).sort_values('SHAP Value', key=abs, ascending=False)

    for _, row in shap_df.head(3).iterrows():
        impact = '🟢 Toward Approval' if row['SHAP Value'] > 0 else '🔴 Toward Denial'
        print(f'  {impact} | {row["Feature"]}: {row["SHAP Value"]:.3f}')

    # ── LIME ──────────────────────────────────────
    print('\n' + '-'*40)
    print('🍋 LIME - Top 3 Reasons:')
    print('-'*40)
    lime_result = explainer_lime.explain_instance(
        instance.values[0], model.predict_proba, num_features=3
    )
    for feat, weight in lime_result.as_list():
        impact = '🟢 Toward Approval' if weight > 0 else '🔴 Toward Denial'
        print(f'  {impact} | {feat}: {weight:.3f}')

    # ── Counterfactual ─────────────────────────────
    if prediction == 0:
        print('\n' + '-'*40)
        print('🎲 Counterfactual - To get APPROVED, change:')
        print('-'*40)
        cf = dice_exp.generate_counterfactuals(
            instance, total_CFs=1, desired_class='opposite'
        )
        cf_result = cf.cf_examples_list[0].final_cfs_df
        for col in feature_names:
            orig_val = instance[col].values[0]
            cf_val = cf_result[col].values[0]
            if round(orig_val, 2) != round(cf_val, 2):
                print(f'  📌 {col}: {orig_val:.2f} → {cf_val:.2f}')
        print('  ✅ Result: LOAN APPROVED!')
    else:
        print('\n✅ Loan is already APPROVED — no changes needed!')

    print('='*60)

print('✅ explain_decision() function ready!')

In [ ]:
# Test on a DENIED case
explain_decision(denied_pos)

In [ ]:
# Test on an APPROVED case
approved_pos = list(X_test.index).index(X_test[y_pred == 1].index[0])
explain_decision(approved_pos)

## 🎯 Summary

| Method | Kya batata hai | Kab use karein |
|--------|---------------|----------------|
| **SHAP** | Har feature ka global + local impact | Model analysis, debugging |
| **LIME** | Is prediction ke liye kya important tha | Client ko explain karna |
| **DiCE** | Kya badlo toh decision badle | User guidance, actionable steps |

---

### 🔥 Key Takeaways:
1. **Credit_History** sabse important feature hai (SHAP + LIME dono agree karte hain)
2. **Counterfactual** user ko exact actionable steps deta hai
3. Teeno methods milke ek **complete, trustworthy XAI system** banate hain

---
*Project by: Your Name | Dataset: Kaggle Loan Prediction*